In [1]:
import os
import pandas as pd
import numpy as np

print("Starting baseline submission script")

PLATFORM = "kaggle"  # change to "local" if running locally
SEED = 2024
np.random.seed(SEED)

if PLATFORM == "local":
    train_path = "./input/hms-harmful-brain-activity-classification/train.csv"
    test_path = "./input/hms-harmful-brain-activity-classification/test.csv"
else:
    train_path = "/kaggle/input/hms-harmful-brain-activity-classification/train.csv"
    test_path = "/kaggle/input/hms-harmful-brain-activity-classification/test.csv"

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

TARGETS = train.columns[-6:]  # seizure, lpd, gpd, lrda, grda, other
print("Targets:", list(TARGETS))

# Compute global class prior
class_votes_sum = train[TARGETS].sum().astype(np.float64) + 1e-12
total_votes = class_votes_sum.sum()
global_prior = (class_votes_sum / total_votes).values  # (n_classes,)

# Patient‑level prior
patient_counts = train.groupby("patient_id")[list(TARGETS)].sum()
patient_sums = patient_counts.sum(axis=1)  # total votes per patient
patient_prior = patient_counts.div(
    patient_sums.replace(0, np.nan), axis=0
)  # (n_patients, n_classes)

# EEG‑level prior
eeg_counts = train.groupby("eeg_id")[list(TARGETS)].sum()
eeg_sums = eeg_counts.sum(axis=1)  # total votes per eeg_id
eeg_prior = eeg_counts.div(eeg_sums.replace(0, np.nan), axis=0)  # (n_eegs, n_classes)

# Lower smoothing factor to give more weight to specific priors
alpha = 5.0

submission = test[["eeg_id"]].copy()

# Align priors with test rows
eeg_prior_matrix = eeg_prior.reindex(
    test["eeg_id"]
).values  # (n_rows, n_classes) with NaNs
patient_prior_matrix = patient_prior.reindex(test["patient_id"]).values

eeg_available = ~np.isnan(eeg_prior_matrix).any(axis=1)

specific_prior = np.zeros((len(test), len(TARGETS)), dtype=np.float64)

specific_prior[eeg_available] = eeg_prior_matrix[eeg_available]

patient_available = ~np.isnan(patient_prior_matrix).any(axis=1)
need_patient = ~eeg_available & patient_available
specific_prior[need_patient] = patient_prior_matrix[need_patient]

remaining = ~(eeg_available | need_patient)
specific_prior[remaining] = global_prior

# Compute vote counts for weighting
patient_vote_counts = patient_sums.reindex(test["patient_id"]).fillna(0).values
eeg_vote_counts = eeg_sums.reindex(test["eeg_id"]).fillna(0).values

row_votes = np.maximum(patient_vote_counts, eeg_vote_counts)

weight = row_votes / (row_votes + alpha)
weight = weight[:, None]  # reshape to (n_rows, 1)

blended = weight * specific_prior + (1.0 - weight) * global_prior  # broadcasting

# Normalise to ensure each row sums to 1
row_sums = blended.sum(axis=1, keepdims=True)
blended = np.where(np.isclose(row_sums, 1.0, atol=1e-12), blended, blended / row_sums)

submission[TARGETS] = blended

# Verify row sums
row_sums_check = submission[TARGETS].sum(axis=1)
assert np.allclose(row_sums_check, 1.0, atol=1e-6), "Row sums are not 1.0"

submission_path = "submission.csv"
submission.to_csv(submission_path, index=False)
print(f"Submission saved to {submission_path}")
print("Submission shape:", submission.shape)
print(
    "First few row sums (should be 1.0):",
    submission[TARGETS].sum(axis=1).head().values,
)

Starting baseline submission script
Targets: ['seizure_vote', 'lpd_vote', 'gpd_vote', 'lrda_vote', 'grda_vote', 'other_vote']
Submission saved to submission.csv
Submission shape: (9850, 7)
First few row sums (should be 1.0): [1. 1. 1. 1. 1.]
